# Premier League Prediction V1

## Configuration Constants

In [28]:
# Imports used across the notebook for data loading, feature engineering, and modeling.
import re
from pathlib import Path
from typing import Optional

import numpy as np
import pandas as pd
from xgboost import XGBClassifier
from sklearn.compose import ColumnTransformer
from sklearn.ensemble import RandomForestClassifier
from sklearn.impute import SimpleImputer
from sklearn.linear_model import LogisticRegression
from sklearn.metrics import accuracy_score, classification_report
from sklearn.model_selection import train_test_split
from sklearn.neural_network import MLPClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import LabelEncoder, OneHotEncoder, StandardScaler
from sklearn.svm import SVC
from sklearn.tree import DecisionTreeClassifier

# Folder that contains the training data files.
DATA_ROOT = Path("../data")

# Set this to True to include the additional merged Premier League copy and the cup-based fatigue data.
USE_ADDITIONAL_DATA = True

In [29]:
ROLLING_WINDOWS = (3, 5, 10)
ROLLING_WINDOW = 5

ROLLING_DECAY = 0.75

COUNTRY_CODES = {
    "eng",
    "es",
    "de",
    "fr",
    "it",
    "nl",
    "pt",
    "ru",
    "ua",
    "gr",
    "be",
    "at",
    "hr",
    "rs",
    "tr",
    "ch",
    "dk",
    "no",
    "se",
    "pl",
    "cz",
    "sk",
}

TEAM_ALIASES = {
    "man city": "manchester city",
    "man utd": "manchester united",
    "manchester utd": "manchester united",
    "spurs": "tottenham hotspur",
    "tottenham": "tottenham hotspur",
    "wolves": "wolverhampton wanderers",
    "leicester": "leicester city",
    "norwich": "norwich city",
    "west ham": "west ham united",
    "newcastle": "newcastle united",
    "brighton": "brighton and hove albion",
}

COMP_WEIGHTS = {
    "ucl": {"group": 1.0, "knockout": 2.0, "late": 3.0},
    "uel": {"group": 0.8, "knockout": 1.5, "late": 2.0},
    "fa": {"early": 0.5, "late": 1.5},
    "carabao": {"early": 0.3, "late": 1.0},
}

TEAM_BOOST_OVERRIDES = {
    "liverpool": 0.40,
    "manchester city": 0.00,
}

CONTEXT_COLS = {
    "Possession": ["Possession"],
    "Shot_Creating_Actions": ["Shot_Creating_Actions", "SCA"],
    "Successful_Dribbles": ["Successful_Dribbles", "Dribbles"],
    "Final_Third_Entries": ["Final_Third_Entries"],
    "Final_Third_Entries_Allowed": ["Final_Third_Entries_Allowed"],
    "Aerial_Battles_Won_Pct": ["Aerial_Battles_Won%", "Aerial_Battles_Won_Pct"],
    "Save_Pct": ["Save%", "Save_Pct"],
    "PPDA": ["PPDA"],
    "Allowed_PPDA": ["Allowed_PPDA"],
}

## Utility Functions

In [30]:
# Print a simple notebook log message.
def log(msg: str) -> None:
    print(f"[INFO] {msg}")


# Find the first column name that matches one of the candidates.
def find_col(df: pd.DataFrame, *candidates: str) -> Optional[str]:
    lower_map = {c.lower(): c for c in df.columns}
    for cand in candidates:
        if cand.lower() in lower_map:
            return lower_map[cand.lower()]
    return None


# Normalize team names so aliases and country prefixes match.
def normalize_team(name) -> str:
    s = re.sub(r"[^a-z0-9\s]", " ", str(name).strip().lower())
    tokens = [t for t in s.split() if t]
    if tokens and tokens[0] in COUNTRY_CODES:
        tokens = tokens[1:]
    if tokens and tokens[-1] in COUNTRY_CODES:
        tokens = tokens[:-1]
    s = " ".join(tokens)
    return TEAM_ALIASES.get(s, s)


# Return the column if it exists, otherwise fill with NaN values.
def col_or_nan(df: pd.DataFrame, col: str) -> pd.Series:
    if col in df.columns:
        return df[col]
    return pd.Series(np.nan, index=df.index)

## Data Loading Functions

In [31]:
# Load the league data, optionally merging the extra copy into it.
def load_premier_league(base: Path) -> pd.DataFrame:
    df = pd.read_csv(base / "premier_league.csv", low_memory=False)
    if USE_ADDITIONAL_DATA:
        extra = base / "additional_data" / "premier_league.csv"
        extra_df = pd.read_csv(extra, low_memory=False)
        common = list(set(df.columns) & set(extra_df.columns))
        df = pd.concat([df[common], extra_df[common]], ignore_index=True)
        df = df.drop_duplicates().copy()
        log("Loaded additional_data/premier_league.csv")
    log("Loaded premier_league.csv")
    log(f"Raw rows loaded: {len(df):,}")
    return df


# Convert the venue/home-away field into a simple h/a label.
def parse_home_away(df: pd.DataFrame) -> pd.Series:
    for col in ("h_a", "side"):
        if col in df.columns:
            return df[col].astype(str).str.lower().str.strip()
    if "Venue" in df.columns:
        return (
            df["Venue"]
            .astype(str)
            .str.lower()
            .str.strip()
            .map({"home": "h", "away": "a"})
        )
    return pd.Series(["a"] * len(df), index=df.index)


# Read the position table and turn it into a team-to-rank lookup.
def load_position_map(path: Path) -> dict[str, float]:
    df = pd.read_csv(path)
    df.columns = df.columns.str.strip()
    team_col = find_col(df, "team", "teamid", "club")
    pos_col = find_col(df, "position", "pos", "rank")

    if not team_col or not pos_col:
        raise ValueError(f"{path} must have Team and Position columns")

    df["team_norm"] = df[team_col].map(normalize_team)
    df["pos"] = pd.to_numeric(df[pos_col], errors="coerce")
    df = df.dropna(subset=["team_norm", "pos"])
    return dict(zip(df["team_norm"], df["pos"]))


# Infer the competition stage from the row text and date.
def _infer_stage(comp: str, stage_text: str, dt: pd.Timestamp) -> str:
    s = str(stage_text).lower()
    if any(
        k in s for k in ["semi", "final", "quarter", "qf", "sf", "r16", "round of 16"]
    ):
        return (
            "late"
            if any(k in s for k in ["semi", "final", "quarter", "qf", "sf"])
            else "knockout"
        )
    if "group" in s:
        return "group"
    month = dt.month if pd.notna(dt) else 1
    if comp in ("ucl", "uel"):
        return (
            "group"
            if month in [9, 10, 11, 12]
            else ("late" if month == 8 else "knockout")
        )
    if comp in ("fa", "carabao"):
        return "late" if month >= 3 else "early"
    return "early"


# Turn cup fixtures into weighted fatigue events for each team.
def load_competition_matches(path: Path, comp: str) -> pd.DataFrame:
    df = pd.read_csv(path, dtype=str)
    df.columns = df.columns.str.strip()

    date_col = find_col(df, "date")
    home_col = find_col(df, "home team", "home_team", "home")
    away_col = find_col(df, "away team", "away_team", "away")
    stage_col = find_col(df, "stage", "round")

    if not all([date_col, home_col, away_col]):
        raise ValueError(f"{path.name} is missing required columns")

    # Support mixed date styles like YYYY-MM-DD, May 30 2015, and April 18-19 2015.
    date_text = df[date_col].astype(str).str.strip()
    date_text = date_text.str.replace(
        r"([A-Za-z]+\s+\d{1,2})-\d{1,2}(\s+\d{4})", r"\1\2", regex=True
    )
    df["_date"] = pd.to_datetime(date_text, errors="coerce")
    df["_stage"] = df[stage_col] if stage_col else ""

    rows = []
    for _, row in df.iterrows():
        if pd.isna(row["_date"]):
            continue
        stage = _infer_stage(comp, row["_stage"], row["_date"])
        weight = COMP_WEIGHTS.get(comp, {}).get(stage, 0.0)
        for col in (home_col, away_col):
            rows.append(
                {
                    "team": normalize_team(row[col]),  # type: ignore
                    "date": row["_date"],
                    "weight": weight,
                }
            )

    return pd.DataFrame(rows, columns=["team", "date", "weight"])

## Basic Feature Engineering

In [32]:
# Attach a fatigue score based on past competition workload.
def attach_fatigue(df: pd.DataFrame, comp_df: pd.DataFrame) -> pd.DataFrame:
    comp_df = comp_df.dropna(subset=["team", "date"]).sort_values("date").copy()
    comp_df["date"] = pd.to_datetime(comp_df["date"], errors="coerce")
    comp_df = comp_df.dropna(subset=["date"])
    comp_df["date"] = comp_df["date"].astype("datetime64[ns]")
    comp_df["weight"] = pd.to_numeric(comp_df["weight"], errors="coerce").fillna(0.0)
    comp_df["cum_weight"] = comp_df.groupby("team")["weight"].transform(
        lambda s: s.cumsum().shift(1).fillna(0.0)
    )

    left = df.sort_values("Date").reset_index().rename(columns={"index": "_orig_idx"})
    left["Team"] = left["Team"].astype(str)
    left["Date"] = pd.to_datetime(left["Date"], errors="coerce")
    left["Date"] = left["Date"].astype("datetime64[ns]")

    right = comp_df.rename(columns={"team": "Team"}).sort_values("date")
    right["Team"] = right["Team"].astype(str)

    def _window_cum_weight(
        left_team: pd.DataFrame, right_team: pd.DataFrame, days: int
    ) -> pd.Series:
        boundary = left_team[["_orig_idx", "Date"]].copy()
        boundary["Date"] = boundary["Date"] - pd.Timedelta(days=days)
        merged = pd.merge_asof(
            boundary.sort_values("Date"),
            right_team[["date", "cum_weight"]].sort_values("date"),
            left_on="Date",
            right_on="date",
            direction="backward",
            allow_exact_matches=False,
        )
        return merged.set_index("_orig_idx")["cum_weight"].sort_index().fillna(0.0)

    left["fatigue_score"] = 0.0
    left["fatigue_7d"] = 0.0
    left["fatigue_14d"] = 0.0
    left["fatigue_30d"] = 0.0
    left["fatigue_recent_share"] = 0.0

    left_valid = left[left["Date"].notna()].sort_values("Date").copy()
    if not left_valid.empty and not right.empty:
        merged_parts = []
        for team, left_team in left_valid.groupby("Team", sort=False):
            right_team = right[right["Team"].eq(team)]
            if right_team.empty:
                continue

            left_team = left_team.sort_values("Date").copy()
            total = _window_cum_weight(left_team, right_team, days=0)
            fatigue_7d = total - _window_cum_weight(left_team, right_team, days=7)
            fatigue_14d = total - _window_cum_weight(left_team, right_team, days=14)
            fatigue_30d = total - _window_cum_weight(left_team, right_team, days=30)

            team_features = pd.DataFrame(
                {
                    "_orig_idx": left_team["_orig_idx"].to_numpy(),
                    "fatigue_7d": fatigue_7d.to_numpy(),
                    "fatigue_14d": fatigue_14d.to_numpy(),
                    "fatigue_30d": fatigue_30d.to_numpy(),
                }
            )
            team_features["fatigue_recent_share"] = np.where(
                team_features["fatigue_30d"] > 0,
                team_features["fatigue_7d"] / team_features["fatigue_30d"],
                0.0,
            )
            team_features["fatigue_score"] = (
                2.0 * team_features["fatigue_7d"]
                + 1.1 * (team_features["fatigue_14d"] - team_features["fatigue_7d"])
                + 0.6 * (team_features["fatigue_30d"] - team_features["fatigue_14d"])
            )
            merged_parts.append(team_features)

        if merged_parts:
            fatigue_df = (
                pd.concat(merged_parts, axis=0).set_index("_orig_idx").sort_index()
            )
            left.loc[
                fatigue_df.index,
                [
                    "fatigue_score",
                    "fatigue_7d",
                    "fatigue_14d",
                    "fatigue_30d",
                    "fatigue_recent_share",
                ],
            ] = fatigue_df[
                [
                    "fatigue_score",
                    "fatigue_7d",
                    "fatigue_14d",
                    "fatigue_30d",
                    "fatigue_recent_share",
                ]
            ].to_numpy()

    df["fatigue_score"] = left.sort_values("_orig_idx")["fatigue_score"].to_numpy()
    df["fatigue_7d"] = left.sort_values("_orig_idx")["fatigue_7d"].to_numpy()
    df["fatigue_14d"] = left.sort_values("_orig_idx")["fatigue_14d"].to_numpy()
    df["fatigue_30d"] = left.sort_values("_orig_idx")["fatigue_30d"].to_numpy()
    df["fatigue_recent_share"] = left.sort_values("_orig_idx")[
        "fatigue_recent_share"
    ].to_numpy()
    log(f"Fatigue scores attached from {len(comp_df):,} competition rows")
    return df


# Build a rolling mean using only earlier matches.
def _lagged_rolling_mean(series: pd.Series, window: int) -> pd.Series:
    return series.shift(1).rolling(window, min_periods=1).mean()


# Build a recency-weighted rolling mean using only earlier matches.
def _lagged_weighted_mean(
    series: pd.Series, window: int, decay: float = ROLLING_DECAY
) -> pd.Series:
    shifted = pd.to_numeric(series, errors="coerce").shift(1)

    def _weighted_mean(values: np.ndarray) -> float:
        values = np.asarray(values, dtype=float)
        valid_mask = np.isfinite(values)
        if not valid_mask.any():
            return float(np.nan)
        values = values[valid_mask]
        weights = np.array(
            [decay ** (len(values) - idx - 1) for idx in range(len(values))],
            dtype=float,
        )
        weights = weights / weights.sum()
        return float(np.dot(values, weights))

    return shifted.rolling(window, min_periods=1).apply(_weighted_mean, raw=True)


# Create lagged rolling features for the main performance columns.
def build_rolling_features(
    df: pd.DataFrame, windows: tuple[int, ...] = ROLLING_WINDOWS
) -> pd.DataFrame:
    df = df.sort_values("Date").copy()
    grp = df.groupby("Team", group_keys=False)

    metrics = {
        "xG": "xG",
        "xGA": "xGA",
        "scored": "_scored",
        "conceded": "_conceded",
        "win_rate": "_win_val",
        "ppda": "_ppda",
    }
    if "_xpts" in df.columns:
        metrics["xpts"] = "_xpts"

    for window in windows:
        for label, column in metrics.items():
            df[f"rolling_{label}_{window}"] = grp[column].transform(
                lambda s, w=window: _lagged_rolling_mean(s, w)
            )
            df[f"weighted_{label}_{window}"] = grp[column].transform(
                lambda s, w=window: _lagged_weighted_mean(s, w)
            )

        df[f"rolling_goal_diff_{window}"] = (
            df[f"rolling_scored_{window}"] - df[f"rolling_conceded_{window}"]
        )
        df[f"weighted_goal_diff_{window}"] = (
            df[f"weighted_scored_{window}"] - df[f"weighted_conceded_{window}"]
        )

    log(f"Rolling features built (windows={windows})")
    return df


# Attach opponent rolling features so the model can see the opposition's form.
def attach_opponent_rolling_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values(["Date", "Opponent"]).copy()
    rolling_cols = [
        col
        for col in out.columns
        if col.startswith("rolling_") or col.startswith("weighted_")
    ]
    if not rolling_cols:
        return out

    source = out[["Team", "Date"] + rolling_cols].copy()
    source = source.rename(
        columns={"Team": "Opponent", **{col: f"opp_{col}" for col in rolling_cols}}
    )
    source = source.sort_values(["Opponent", "Date"])

    opp_cols = [f"opp_{col}" for col in rolling_cols]
    out["_orig_idx"] = out.index
    merged_parts = []
    for opponent, left_team in out.groupby("Opponent", sort=False):
        right_team = source[source["Opponent"].eq(opponent)]
        left_team = left_team.sort_values("Date").copy()
        if right_team.empty:
            for col in opp_cols:
                left_team[col] = np.nan
            merged_parts.append(left_team)
            continue

        merged = pd.merge_asof(
            left_team,
            right_team.drop(columns=["Opponent"]),
            left_on="Date",
            right_on="Date",
            direction="backward",
            allow_exact_matches=False,
        )
        merged_parts.append(merged)

    out = pd.concat(merged_parts, axis=0)
    out = out.set_index("_orig_idx").sort_index()

    log("Opponent rolling features attached")
    return out


# Add league position features for the team and opponent.
def attach_position_features(df: pd.DataFrame, pos_map: dict) -> pd.DataFrame:
    df = df.copy()
    df["team_position"] = df["Team"].map(pos_map)
    df["opponent_position"] = df["Opponent"].map(pos_map)
    df["position_gap"] = df["opponent_position"] - df["team_position"]
    return df


# Add a head-to-head win rate for the two teams in a fixture.
def attach_h2h_features(df: pd.DataFrame) -> pd.DataFrame:
    df = df.sort_values("Date").copy()
    df["_h2h_key"] = df[["Team", "Opponent"]].apply(
        lambda r: "|".join(sorted([r["Team"], r["Opponent"]])), axis=1
    )
    df["_h2h_win"] = (df["Result"] == "w").astype(float)
    df["_h2h_goal_diff"] = pd.to_numeric(
        col_or_nan(df, "Scored"), errors="coerce"
    ) - pd.to_numeric(col_or_nan(df, "Conceded"), errors="coerce")
    df["_h2h_points"] = df["Result"].map({"w": 3.0, "d": 1.0, "l": 0.0})

    grouped = df.groupby("_h2h_key")
    df["h2h_win_rate"] = (
        grouped["_h2h_win"]
        .transform(lambda s: s.shift(1).expanding().mean())
        .fillna(0.5)
    )
    df["h2h_goal_diff"] = (
        grouped["_h2h_goal_diff"]
        .transform(lambda s: s.shift(1).expanding().mean())
        .fillna(0.0)
    )
    df["h2h_match_count"] = grouped.cumcount().astype(float)
    df["h2h_points_per_game"] = (
        grouped["_h2h_points"]
        .transform(lambda s: s.shift(1).expanding().mean())
        .fillna(1.0)
    )
    df.drop(
        columns=["_h2h_key", "_h2h_win", "_h2h_goal_diff", "_h2h_points"], inplace=True
    )
    log("H2H features attached")
    return df


# Copy any matching context columns into the canonical names.
def attach_context_features(df: pd.DataFrame) -> tuple[pd.DataFrame, list[str]]:
    added = []
    for canonical, candidates in CONTEXT_COLS.items():
        src = find_col(df, *candidates)
        if src is not None:
            df[canonical] = pd.to_numeric(df[src], errors="coerce")
            added.append(canonical)
    if added:
        log(f"Context features: {added}")
    return df, added

## Form and Match Context Calculations

In [33]:
# Summarize recent results and scoring form for a team using only matches before current_date.
def calculate_recent_form(
    df: pd.DataFrame, team: str, current_date: pd.Timestamp, window: int = 5
) -> dict:
    team_history = df[(df["Team"].eq(team)) & (df["Date"] < current_date)].sort_values(
        "Date"
    )
    if len(team_history) < 1:
        return {
            "points_per_game": 0.0,
            "streak": 0,
            "recent_xg": 0.0,
            "recent_goals": 0.0,
        }

    recent = team_history.tail(window)

    points = recent["Result"].map({"w": 1.0, "d": 0.5, "l": 0.0}).sum()
    points_per_game = points / len(recent) if len(recent) > 0 else 0.0
    recent_xg = pd.to_numeric(col_or_nan(recent, "xG"), errors="coerce").mean()
    recent_goals = pd.to_numeric(col_or_nan(recent, "Scored"), errors="coerce").mean()

    streak = 0
    for result in recent["Result"].iloc[::-1]:
        if result == "w":
            streak += 1
        elif result == "l":
            streak -= 1
        else:
            break

    return {
        "points_per_game": points_per_game,
        "streak": streak,
        "recent_xg": recent_xg if pd.notna(recent_xg) else 0.0,
        "recent_goals": recent_goals if pd.notna(recent_goals) else 0.0,
    }


# Estimate the opponent strength from recent results before current_date.
def calculate_opponent_strength(
    df: pd.DataFrame, opponent: str, current_date: pd.Timestamp, window: int = 10
) -> float:
    opp_history = df[
        (df["Team"].eq(opponent)) & (df["Date"] < current_date)
    ].sort_values("Date")
    if opp_history.empty:
        return 0.5
    opp_df = opp_history.tail(window)
    opp_df["_win_val"] = opp_df["Result"].map({"w": 1.0, "d": 0.5, "l": 0.0})
    return float(opp_df["_win_val"].mean())


# Turn league positions into a simple fixture importance score.
def calculate_fixture_importance(team_pos: float, opponent_pos: float) -> float:
    if pd.isna(team_pos) or pd.isna(opponent_pos):
        return 0.5

    importance = 0.0
    if team_pos <= 2:
        importance += 0.7
    if team_pos >= 16:
        importance += 0.8
    if 3 <= team_pos <= 7:
        importance += 0.5

    pos_gap = abs(team_pos - opponent_pos)
    if opponent_pos <= 6:
        importance += 0.4
    if pos_gap <= 2:
        importance += 0.3

    return min(1.0, importance)

## Advanced Feature Engineering

In [34]:
DEFAULT_POSSESSION_PCT = 50.0
DEFAULT_PPDA = 6.0


# Read a numeric value from a row with a fallback default.
def _row_numeric(row: pd.Series, key: str, default: float = np.nan) -> float:
    value = pd.to_numeric(pd.Series([row.get(key, default)]), errors="coerce").iloc[0]
    if pd.isna(value):
        return default
    return float(value)


# Build simple context features from recent form and opponent strength.
def _match_context_features_for_row(
    row: pd.Series, rolling_window: int, recent_form: dict, opponent_strength: float
) -> dict[str, float]:
    weighted_points = _row_numeric(row, f"weighted_win_rate_{rolling_window}")
    if pd.notna(weighted_points):
        rolling_points = weighted_points * 3.0
    else:
        rolling_points = _row_numeric(row, f"rolling_win_rate_{rolling_window}")
        if pd.notna(rolling_points):
            rolling_points *= 3.0
        else:
            rolling_points = recent_form["points_per_game"] * 3.0

    xg_edge = _row_numeric(row, f"weighted_xG_{rolling_window}") - _row_numeric(
        row, f"opp_weighted_xG_{rolling_window}"
    )
    goal_diff_edge = _row_numeric(
        row, f"weighted_goal_diff_{rolling_window}"
    ) - _row_numeric(row, f"opp_weighted_goal_diff_{rolling_window}")
    win_rate_edge = _row_numeric(
        row, f"weighted_win_rate_{rolling_window}"
    ) - _row_numeric(row, f"opp_weighted_win_rate_{rolling_window}")

    return {
        "form_vs_strength": recent_form["points_per_game"] - opponent_strength,
        "momentum": rolling_points + recent_form["streak"],
        "xg_edge": 0.0 if pd.isna(xg_edge) else float(xg_edge),
        "goal_diff_edge": 0.0 if pd.isna(goal_diff_edge) else float(goal_diff_edge),
        "win_rate_edge": 0.0 if pd.isna(win_rate_edge) else float(win_rate_edge),
    }


# Estimate how referee tendencies affect the match.
def _referee_pressure_impact_for_row(row: pd.Series) -> float:
    is_home = str(row.get("home_advantage", "a")).lower() == "h"
    foul_col = "Home_Fouls" if is_home else "Away_Fouls"

    team_fouls = _row_numeric(row, foul_col, default=0.0)
    ppda_value = _row_numeric(row, "PPDA", default=DEFAULT_PPDA)
    referee_bias = _row_numeric(row, "Referee_Bias_Score", default=0.0)

    aggression = team_fouls + max(0.0, 8.0 - ppda_value)
    return referee_bias * (1.0 + aggression / 10.0)


# Add match-context engineered features to the data.
def attach_match_context_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values("Date").copy()
    w = ROLLING_WINDOW
    feature_rows = []

    for _, row in out.iterrows():
        current_date = row["Date"]
        history = out[out["Date"] < current_date]

        team = row["Team"]
        opponent = row["Opponent"]

        recent_form = calculate_recent_form(history, team, current_date, window=5)
        opp_strength = calculate_opponent_strength(
            history, opponent, current_date, window=10
        )

        row_features = _match_context_features_for_row(
            row, w, recent_form, opp_strength
        )
        row_features["referee_pressure_impact"] = _referee_pressure_impact_for_row(row)

        feature_rows.append(row_features)

    engineered = pd.DataFrame(feature_rows, index=out.index)
    out = pd.concat([out, engineered], axis=1)
    log("Match-context features attached")
    return out


# Build referee-related bias features from card and penalty patterns.
def attach_referee_features(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values("Date").copy()
    out["_ref"] = (
        out["Referee"].astype(str).str.lower().str.strip().replace("", "unknown")
    )
    home_mask = out["home_advantage"].astype(str).str.lower().eq("h")

    home_y = pd.to_numeric(col_or_nan(out, "Home_Yellow"), errors="coerce")
    home_r = pd.to_numeric(col_or_nan(out, "Home_Red"), errors="coerce")
    away_y = pd.to_numeric(col_or_nan(out, "Away_Yellow"), errors="coerce")
    away_r = pd.to_numeric(col_or_nan(out, "Away_Red"), errors="coerce")

    home_cards = home_y.fillna(0.0) + 2.0 * home_r.fillna(0.0)
    away_cards = away_y.fillna(0.0) + 2.0 * away_r.fillna(0.0)

    team_cards = np.where(home_mask, home_cards, away_cards)
    opp_cards = np.where(home_mask, away_cards, home_cards)

    pk_for = pd.to_numeric(col_or_nan(out, "PK"), errors="coerce").fillna(0.0)
    pk_against = pd.to_numeric(col_or_nan(out, "PK_Allowed"), errors="coerce").fillna(
        0.0
    )

    raw_signal = (opp_cards - team_cards) + 2.0 * (pk_for - pk_against)
    raw_signal = pd.Series(raw_signal, index=out.index).fillna(0.0)

    std = raw_signal.std()
    out["_ref_signal_norm"] = (
        (raw_signal - raw_signal.mean()) / std if pd.notna(std) and std > 0 else 0.0
    )

    out["Referee_Bias_Score"] = (
        out.groupby(["_ref", "Team"])["_ref_signal_norm"]
        .transform(lambda s: s.shift(1).expanding().mean())
        .fillna(0.0)
    )

    relegation_mask = out["team_position"].fillna(10) >= 16
    title_mask = out["team_position"].fillna(10) <= 2
    situation_multiplier = np.where(relegation_mask | title_mask, 1.3, 1.0)
    out["Referee_Bias_Score"] = out["Referee_Bias_Score"] * situation_multiplier

    out.drop(columns=["_ref", "_ref_signal_norm"], inplace=True)
    log("Referee bias features attached")
    return out


# Build a motivation score from form, position, and fixture context.
def attach_dynamic_motivation(df: pd.DataFrame) -> pd.DataFrame:
    out = df.sort_values("Date").copy()
    motivation_scores = []

    def _position_component(team_pos: float) -> float:
        if pd.isna(team_pos):
            return 0.0
        if team_pos == 1:
            return 0.4
        if team_pos <= 5:
            return 0.25
        if team_pos <= 8:
            return 0.15
        if team_pos >= 16:
            return 0.5
        return 0.0

    for _, row in out.iterrows():
        current_date = row["Date"]
        history = out[out["Date"] < current_date]

        team = row["Team"]
        opp = row["Opponent"]
        team_pos = row.get("team_position", np.nan)
        opp_pos = row.get("opponent_position", np.nan)

        form = calculate_recent_form(history, team, current_date, window=5)
        form_score = 0.3 * form["points_per_game"]
        form_score += 0.05 * max(form["streak"], -3)

        position_score = _position_component(team_pos)
        opp_strength = calculate_opponent_strength(
            history, opp, current_date, window=10
        )
        opp_score = 0.15 * opp_strength
        fixture_score = 0.2 * calculate_fixture_importance(team_pos, opp_pos)
        home_score = 0.05 if str(row.get("home_advantage", "a")).lower() == "h" else 0.0

        boost = TEAM_BOOST_OVERRIDES.get(team.lower(), 0.0)

        total_motivation = (
            form_score + position_score + opp_score + fixture_score + home_score + boost
        )
        motivation_scores.append(min(1.0, max(0.0, total_motivation)))

    out["Motivation_Score"] = motivation_scores
    log("Dynamic motivation scores calculated")
    return out

## Split Utility (Leakage-Safe Group Split)

In [35]:
# Make sure the validation and test splits are sensible.
def _validate_split_fractions(val_frac: float, test_frac: float) -> None:
    if val_frac <= 0 or test_frac <= 0:
        raise ValueError("val_frac and test_frac must both be > 0")
    if val_frac + test_frac >= 1.0:
        raise ValueError("val_frac + test_frac must be < 1.0")


# Build a stable group id for each fixture match-up.
def make_match_group_id(df: pd.DataFrame) -> pd.Series:
    t1 = df["Team"].str.lower().str.strip()
    t2 = df["Opponent"].str.lower().str.strip()
    lo = t1.where(t1 <= t2, t2)
    hi = t2.where(t1 <= t2, t1)
    date_str = (
        pd.to_datetime(df["Date"], errors="coerce").dt.strftime("%Y-%m-%d").fillna("NA")
    )
    return date_str + "|" + lo + "|" + hi


# Chronological split by unique fixture dates (strictly disjoint date blocks).
def time_based_split(
    df: pd.DataFrame,
    val_frac: float = 0.20,
    test_frac: float = 0.10,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame, pd.DataFrame]:
    _validate_split_fractions(val_frac, test_frac)

    ordered = df.copy()
    ordered["Date"] = pd.to_datetime(ordered["Date"], errors="coerce")
    ordered = (
        ordered[ordered["Date"].notna()].sort_values("Date").reset_index(drop=True)
    )

    if ordered.empty:
        raise ValueError("No dated rows available for chronological splitting")

    unique_dates = pd.Index(ordered["Date"].dropna().unique()).sort_values()
    n_dates = len(unique_dates)
    if n_dates < 3:
        raise ValueError("Need at least 3 unique dates for train/validation/test")

    n_test_dates = max(1, int(round(n_dates * test_frac)))
    n_val_dates = max(1, int(round(n_dates * val_frac)))
    n_train_dates = n_dates - n_val_dates - n_test_dates

    if n_train_dates < 1:
        raise ValueError("Split fractions leave no dates for training")

    train_dates = set(unique_dates[:n_train_dates])
    val_dates = set(unique_dates[n_train_dates : n_train_dates + n_val_dates])
    test_dates = set(unique_dates[n_train_dates + n_val_dates :])

    train_df = ordered[ordered["Date"].isin(train_dates)].copy()
    val_df = ordered[ordered["Date"].isin(val_dates)].copy()
    test_df = ordered[ordered["Date"].isin(test_dates)].copy()

    if train_df.empty or val_df.empty or test_df.empty:
        raise ValueError("One of the chronological splits is empty")

    # Keep temporal boundaries strict while randomizing row order within each split.
    train_df = train_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    val_df = val_df.sample(frac=1, random_state=random_state).reset_index(drop=True)
    test_df = test_df.sample(frac=1, random_state=random_state).reset_index(drop=True)

    train_max_date = pd.to_datetime(train_df["Date"], errors="coerce").max()
    val_min_date = pd.to_datetime(val_df["Date"], errors="coerce").min()
    val_max_date = pd.to_datetime(val_df["Date"], errors="coerce").max()
    test_min_date = pd.to_datetime(test_df["Date"], errors="coerce").min()

    if train_max_date >= val_min_date:
        raise ValueError(
            "Chronological split violation: training dates overlap validation"
        )
    if val_max_date >= test_min_date:
        raise ValueError("Chronological split violation: validation dates overlap test")

    return train_df, val_df, test_df


# Backward-compatible alias now routed to chronological splitting.
def stratified_group_split(
    df: pd.DataFrame,
    val_frac: float = 0.20,
    test_frac: float = 0.10,
    random_state: int = 42,
):
    return time_based_split(
        df, val_frac=val_frac, test_frac=test_frac, random_state=random_state
    )


# Backward-compatible helper for train/test usage.
def group_train_test_split(
    df: pd.DataFrame,
    test_frac: float = 0.20,
    random_state: int = 42,
) -> tuple[pd.DataFrame, pd.DataFrame]:
    train_df, _, test_df = time_based_split(
        df,
        val_frac=1e-6,
        test_frac=test_frac,
        random_state=random_state,
    )
    return train_df, test_df

## Training Workflow Functions

In [36]:
# Clean the raw league columns into a model-ready base table.
def _normalize_base_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["Date"] = pd.to_datetime(out["Date"], errors="coerce")
    out["Result"] = out["Result"].astype(str).str.lower().str.strip()
    out["Team"] = out["Team"].map(normalize_team)
    out["Opponent"] = out["Opponent"].map(normalize_team)
    out["xG"] = pd.to_numeric(out["xG"], errors="coerce")
    out["xGA"] = pd.to_numeric(out["xGA"], errors="coerce")
    out["home_advantage"] = (
        parse_home_away(out).replace({"home": "h", "away": "a"}).fillna("a")
    )
    return out


# Add helper columns used to build rolling features.
def _add_rolling_helper_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out["_scored"] = pd.to_numeric(col_or_nan(out, "Scored"), errors="coerce")
    out["_conceded"] = pd.to_numeric(col_or_nan(out, "Conceded"), errors="coerce")
    out["_win_val"] = out["Result"].map({"w": 1.0, "d": 0.5, "l": 0.0})
    out["_ppda"] = pd.to_numeric(
        out.get("PPDA", pd.Series(dtype=float)), errors="coerce"
    )
    if "xpts" in out.columns:
        out["_xpts"] = pd.to_numeric(out["xpts"], errors="coerce")
    return out


# Keep only rows that can be used for training.
def _filter_trainable_rows(df: pd.DataFrame) -> pd.DataFrame:
    filtered = df[
        df["Date"].notna()
        & df["Result"].isin(["w", "d", "l"])
        & df["Team"].notna()
        & df["Opponent"].notna()
    ].copy()
    log(f"Rows after filtering: {len(filtered):,}")
    return filtered


# Load competition fixtures only when the toggle is enabled.
def _load_competition_events(data_root: Path) -> pd.DataFrame:
    if not USE_ADDITIONAL_DATA:
        return pd.DataFrame(columns=["team", "date", "weight"])
    additional = data_root / "additional_data"
    return pd.concat(
        [
            load_competition_matches(additional / "champion_league.csv", "ucl"),
            load_competition_matches(additional / "europa_league.csv", "uel"),
            load_competition_matches(additional / "fa_cup.csv", "fa"),
            load_competition_matches(additional / "carabao.csv", "carabao"),
        ],
        ignore_index=True,
    )


# Remove temporary helper columns after rolling features are built.
def _drop_rolling_helper_columns(df: pd.DataFrame) -> pd.DataFrame:
    out = df.copy()
    out.drop(
        columns=["_scored", "_conceded", "_win_val", "_ppda", "_xpts"],
        errors="ignore",
        inplace=True,
    )
    return out


# Load and lightly clean rows before any split-aware feature engineering.
def load_base_dataset(data_root: Path) -> pd.DataFrame:
    df = load_premier_league(data_root)
    df = _normalize_base_columns(df)
    df = _filter_trainable_rows(df)
    return df


COMP_EVENTS_CACHE: Optional[pd.DataFrame] = None
POSITION_MAP_CACHE: Optional[dict[str, float]] = None


# Build leak-safe features for a split using only past history + split rows.
def engineer_features(
    df: pd.DataFrame, history_df: Optional[pd.DataFrame] = None
) -> pd.DataFrame:
    if df.empty:
        return df.copy()

    current = _add_rolling_helper_columns(df.copy())
    current = current.sort_values("Date").copy()

    if history_df is not None and not history_df.empty:
        history = _add_rolling_helper_columns(history_df.copy())
        history = history.sort_values("Date").copy()

        # Reindex history so full.loc(current.index) always selects only split rows.
        history.index = pd.RangeIndex(
            start=len(current), stop=len(current) + len(history)
        )
        full = pd.concat([history, current], axis=0, sort=False)
    else:
        full = current.copy()

    global COMP_EVENTS_CACHE, POSITION_MAP_CACHE
    if COMP_EVENTS_CACHE is None:
        COMP_EVENTS_CACHE = _load_competition_events(DATA_ROOT)
    if POSITION_MAP_CACHE is None:
        POSITION_MAP_CACHE = load_position_map(
            DATA_ROOT / "league_position_after20.csv"
        )

    comp_df = COMP_EVENTS_CACHE
    pos_map = POSITION_MAP_CACHE

    def _apply_with_index_preservation(func, frame: pd.DataFrame, *args):
        idx = frame.index
        updated = func(frame, *args)
        return updated.reindex(idx)

    full = _apply_with_index_preservation(attach_fatigue, full, comp_df)
    full = _apply_with_index_preservation(build_rolling_features, full)
    full = _apply_with_index_preservation(attach_opponent_rolling_features, full)
    full = _apply_with_index_preservation(attach_position_features, full, pos_map)
    full = _apply_with_index_preservation(attach_h2h_features, full)
    _idx_before_context = full.index
    full, _ = attach_context_features(full)
    full = full.reindex(_idx_before_context)
    full = _apply_with_index_preservation(attach_referee_features, full)
    full = _apply_with_index_preservation(attach_match_context_features, full)
    full = _apply_with_index_preservation(attach_dynamic_motivation, full)
    full = _drop_rolling_helper_columns(full)

    missing_idx = current.index.difference(full.index)
    if len(missing_idx) > 0:
        raise ValueError(
            "Feature engineering lost split rows during processing: "
            + ", ".join(map(str, missing_idx[:10]))
        )

    return full.loc[current.index].copy()


# Decide which engineered columns are used as model features.
def select_feature_columns(df: pd.DataFrame):
    window_cols = []
    for window in ROLLING_WINDOWS:
        window_cols.extend(
            [
                f"rolling_xG_{window}",
                f"rolling_xGA_{window}",
                f"rolling_xpts_{window}",
                f"rolling_ppda_{window}",
                f"rolling_scored_{window}",
                f"rolling_conceded_{window}",
                f"rolling_win_rate_{window}",
                f"rolling_goal_diff_{window}",
                f"weighted_xG_{window}",
                f"weighted_xGA_{window}",
                f"weighted_xpts_{window}",
                f"weighted_ppda_{window}",
                f"weighted_scored_{window}",
                f"weighted_conceded_{window}",
                f"weighted_win_rate_{window}",
                f"weighted_goal_diff_{window}",
                f"opp_rolling_xG_{window}",
                f"opp_rolling_xGA_{window}",
                f"opp_rolling_xpts_{window}",
                f"opp_rolling_ppda_{window}",
                f"opp_rolling_scored_{window}",
                f"opp_rolling_conceded_{window}",
                f"opp_rolling_win_rate_{window}",
                f"opp_rolling_goal_diff_{window}",
                f"opp_weighted_xG_{window}",
                f"opp_weighted_xGA_{window}",
                f"opp_weighted_xpts_{window}",
                f"opp_weighted_ppda_{window}",
                f"opp_weighted_scored_{window}",
                f"opp_weighted_conceded_{window}",
                f"opp_weighted_win_rate_{window}",
                f"opp_weighted_goal_diff_{window}",
            ]
        )

    base_numeric_cols = [
        "fatigue_score",
        "fatigue_7d",
        "fatigue_14d",
        "fatigue_30d",
        "fatigue_recent_share",
        "team_position",
        "opponent_position",
        "position_gap",
    ]
    context_keep_cols = [
        "Possession",
        "Shot_Creating_Actions",
        "SCA",
        "Successful_Dribbles",
        "Dribbles",
        "Final_Third_Entries",
        "Final_Third_Entries_Allowed",
        "Aerial_Battles_Won_Pct",
        "Aerial_Battles_Won%",
        "Save_Pct",
        "Save%",
        "PPDA",
        "Allowed_PPDA",
    ]
    psych_cols = [
        "Referee_Bias_Score",
        "Motivation_Score",
        "referee_pressure_impact",
    ]
    interaction_cols = [
        "h2h_win_rate",
        "h2h_goal_diff",
        "h2h_match_count",
        "h2h_points_per_game",
        "form_vs_strength",
        "momentum",
        "xg_edge",
        "goal_diff_edge",
        "win_rate_edge",
    ]

    numeric_cols = [
        c
        for c in (
            base_numeric_cols
            + window_cols
            + context_keep_cols
            + psych_cols
            + interaction_cols
        )
        if c in df.columns
    ]
    cat_cols = ["home_advantage"]
    all_feat_cols = numeric_cols + cat_cols

    log(
        f"Features: {len(all_feat_cols)} total (numeric={len(numeric_cols)}, categorical={len(cat_cols)})"
    )
    return numeric_cols, cat_cols, all_feat_cols


# Fail fast if a feature is a direct copy or exact numeric encoding of the target.
def assert_no_perfect_target_leakage(X: pd.DataFrame, y: pd.Series) -> None:
    target = y.astype(str).str.lower().str.strip()
    leaked_cols = []

    target_codes = pd.Categorical(target).codes
    target_code_series = pd.Series(target_codes, index=y.index, dtype=float)

    for col in X.columns:
        feature = X[col]
        if feature.dtype == object or pd.api.types.is_string_dtype(feature):
            normalized = feature.astype(str).str.lower().str.strip()
            if normalized.equals(target):
                leaked_cols.append(col)
        elif pd.api.types.is_numeric_dtype(feature):
            numeric_feature = pd.to_numeric(feature, errors="coerce")
            aligned = pd.DataFrame(
                {"feature": numeric_feature, "target_code": target_code_series}
            ).dropna()
            if aligned.empty:
                continue
            unique_feature_values = np.sort(aligned["feature"].unique())
            unique_target_values = np.sort(aligned["target_code"].unique())
            if np.array_equal(unique_feature_values, unique_target_values):
                leaked_cols.append(col)

    if leaked_cols:
        raise ValueError(
            "Potential leakage detected: these features directly encode the target: "
            + ", ".join(leaked_cols)
        )


# Split the engineered dataframe into inputs and labels.
def split_features_and_target(
    df: pd.DataFrame,
    feature_cols: list[str],
    val_frac: float = 0.20,
    test_frac: float = 0.10,
):
    train_df, val_df, test_df = stratified_group_split(
        df, val_frac=val_frac, test_frac=test_frac
    )

    X_train, y_train = train_df[feature_cols], train_df["Result"]
    X_val, y_val = val_df[feature_cols], val_df["Result"]
    X_test, y_test = test_df[feature_cols], test_df["Result"]
    return X_train, y_train, X_val, y_val, X_test, y_test


# Print accuracy and classification metrics for a fitted model.
def print_model_report(
    model_name: str,
    model,
    X_val: pd.DataFrame,
    y_val: pd.Series,
    X_test: pd.DataFrame,
    y_test: pd.Series,
    include_feature_importance: bool = False,
):
    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)
    val_acc = accuracy_score(y_val, y_val_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    sep = "=" * 62
    print(f"\n{sep}")
    print(f"  {model_name}")
    print(sep)
    print(f"  Validation Accuracy : {val_acc:.4f}")
    print(f"  Test Accuracy       : {test_acc:.4f}")
    print(f"\nValidation Report:\n{classification_report(y_val, y_val_pred, digits=4)}")
    print(f"Test Report:\n{classification_report(y_test, y_test_pred, digits=4)}")

    if include_feature_importance:
        clf = model.named_steps.get("clf")
        preprocess = model.named_steps.get("preprocess")
        if (
            clf is not None
            and preprocess is not None
            and hasattr(clf, "feature_importances_")
        ):
            feat_names = [
                name.split("__", 1)[-1] for name in preprocess.get_feature_names_out()
            ]
            importance_df = (
                pd.DataFrame(
                    {"feature": feat_names, "importance": clf.feature_importances_}
                )
                .sort_values("importance", ascending=False)
                .reset_index(drop=True)
            )
            print("Top 15 feature importances:")
            print(importance_df.head(15).to_string(index=False))

    return {
        "model": model_name,
        "validation_accuracy": val_acc,
        "test_accuracy": test_acc,
    }

## Model Builders and Training Run

In [37]:
# Build the preprocessing pipeline for numeric and categorical columns.
def build_preprocessor(
    numeric_features: list[str], categorical_features: list[str]
) -> ColumnTransformer:
    return ColumnTransformer(
        transformers=[
            ("num", SimpleImputer(strategy="median"), numeric_features),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                categorical_features,
            ),
        ]
    )


# Build a random forest classification pipeline.
def build_random_forest_pipeline(
    numeric_features: list[str], categorical_features: list[str], n_estimators
) -> Pipeline:
    preprocessor = build_preprocessor(numeric_features, categorical_features)
    clf = RandomForestClassifier(
        n_estimators=n_estimators, class_weight="balanced", random_state=42, n_jobs=-1
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("clf", clf)])


# Build a decision tree classification pipeline.
def build_decision_tree_pipeline(
    numeric_features: list[str], categorical_features: list[str]
) -> Pipeline:
    preprocessor = build_preprocessor(numeric_features, categorical_features)
    clf = DecisionTreeClassifier(
        max_depth=12, min_samples_leaf=5, class_weight="balanced", random_state=42
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("clf", clf)])


# Build a logistic regression classification pipeline.
def build_logistic_regression_pipeline(
    numeric_features: list[str], categorical_features: list[str]
) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                categorical_features,
            ),
        ]
    )
    clf = LogisticRegression(
        max_iter=3000, class_weight="balanced", random_state=42, solver="lbfgs"
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("clf", clf)])


# Build an XGBoost classification pipeline.
def build_xgboost_pipeline(
    numeric_features: list[str], categorical_features: list[str], n_estimators
) -> Pipeline:
    preprocessor = build_preprocessor(numeric_features, categorical_features)
    clf = XGBClassifier(
        n_estimators=n_estimators,
        max_depth=6,
        learning_rate=0.05,
        subsample=0.9,
        colsample_bytree=0.9,
        objective="multi:softprob",
        eval_metric="mlogloss",
        random_state=42,
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("clf", clf)])


# Build an SVM classification pipeline.
def build_svm_pipeline(
    numeric_features: list[str], categorical_features: list[str]
) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                categorical_features,
            ),
        ]
    )
    clf = SVC(
        kernel="rbf", C=1.0, gamma="scale", class_weight="balanced", random_state=42
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("clf", clf)])


# Build a neural network classification pipeline.
def build_mlp_pipeline(
    numeric_features: list[str], categorical_features: list[str]
) -> Pipeline:
    preprocessor = ColumnTransformer(
        transformers=[
            (
                "num",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="median")),
                        ("scaler", StandardScaler()),
                    ]
                ),
                numeric_features,
            ),
            (
                "cat",
                Pipeline(
                    [
                        ("imputer", SimpleImputer(strategy="most_frequent")),
                        (
                            "onehot",
                            OneHotEncoder(handle_unknown="ignore", sparse_output=False),
                        ),
                    ]
                ),
                categorical_features,
            ),
        ]
    )
    clf = MLPClassifier(
        hidden_layer_sizes=(64, 32),
        activation="relu",
        alpha=1e-4,
        learning_rate_init=1e-3,
        max_iter=600,
        random_state=42,
    )
    return Pipeline(steps=[("preprocess", preprocessor), ("clf", clf)])


raw_df = load_base_dataset(DATA_ROOT)

# Strict chronological split on raw rows before feature engineering.
train_raw_df, val_raw_df, test_raw_df = time_based_split(
    raw_df, val_frac=0.20, test_frac=0.20, random_state=42
)

# Engineer each split with only past information available at prediction time.
train_df = engineer_features(train_raw_df)
val_df = engineer_features(val_raw_df, history_df=train_raw_df)
test_history_df = pd.concat([train_raw_df, val_raw_df], axis=0, ignore_index=True)
test_df = engineer_features(test_raw_df, history_df=test_history_df)

numeric_cols, cat_cols, all_feat_cols = select_feature_columns(train_df)

X_train, y_train = train_df[all_feat_cols], train_df["Result"]
X_val, y_val = val_df[all_feat_cols], val_df["Result"]
X_test, y_test = test_df[all_feat_cols], test_df["Result"]

assert_no_perfect_target_leakage(X_train, y_train)

results = []

# Train and evaluate classical models.
model_specs = [
    (
        "LogisticRegression (linear model)",
        build_logistic_regression_pipeline(numeric_cols, cat_cols),
    ),
    (
        "DecisionTreeClassifier (rule-based)",
        build_decision_tree_pipeline(numeric_cols, cat_cols),
    ),
    (
        "RandomForestClassifier (bagging ensemble)",
        build_random_forest_pipeline(numeric_cols, cat_cols, n_estimators=200),
    ),
    (
        "SVM (margin-based)",
        build_svm_pipeline(numeric_cols, cat_cols),
    ),
    (
        "MLPClassifier (neural network)",
        build_mlp_pipeline(numeric_cols, cat_cols),
    ),
]

for model_name, model in model_specs:
    model.fit(X_train, y_train)

    y_val_pred = model.predict(X_val)
    y_test_pred = model.predict(X_test)

    val_acc = accuracy_score(y_val, y_val_pred)
    test_acc = accuracy_score(y_test, y_test_pred)

    print("\n" + "=" * 62)
    print(f"  {model_name}")
    print("=" * 62)
    print(f"  Validation Accuracy : {val_acc:.4f}")
    print(f"  Test Accuracy       : {test_acc:.4f}")
    print(f"\nValidation Report:\n{classification_report(y_val, y_val_pred, digits=4)}")
    print(f"Test Report:\n{classification_report(y_test, y_test_pred, digits=4)}")

    results.append(
        {
            "model": model_name,
            "validation_accuracy": val_acc,
            "test_accuracy": test_acc,
        }
    )

# Train and evaluate XGBoost (requires encoded labels).
label_encoder = LabelEncoder()
y_train_enc = np.asarray(label_encoder.fit_transform(y_train), dtype=int)
y_val_enc = np.asarray(label_encoder.transform(y_val), dtype=int)
y_test_enc = np.asarray(label_encoder.transform(y_test), dtype=int)

xgb_model = build_xgboost_pipeline(numeric_cols, cat_cols, n_estimators=200)
xgb_model.fit(X_train, np.asarray(y_train_enc, dtype=np.int64))

y_val_pred_enc = xgb_model.predict(X_val)
y_test_pred_enc = xgb_model.predict(X_test)

val_acc = accuracy_score(y_val_enc, y_val_pred_enc)
test_acc = accuracy_score(y_test_enc, y_test_pred_enc)

print("\n" + "=" * 62)
print("  XGBoostClassifier (boosting)")
print("=" * 62)
print(f"  Validation Accuracy : {val_acc:.4f}")
print(f"  Test Accuracy       : {test_acc:.4f}")
print(
    f"\nValidation Report:\n{classification_report(y_val_enc, y_val_pred_enc, target_names=label_encoder.classes_, digits=4)}"
)
print(
    f"Test Report:\n{classification_report(y_test_enc, y_test_pred_enc, target_names=label_encoder.classes_, digits=4)}"
)

results.append(
    {
        "model": "XGBoostClassifier (boosting)",
        "validation_accuracy": val_acc,
        "test_accuracy": test_acc,
    }
)

[INFO] Loaded additional_data/premier_league.csv
[INFO] Loaded premier_league.csv
[INFO] Raw rows loaded: 3,076
[INFO] Rows after filtering: 3,076
[INFO] Fatigue scores attached from 920 competition rows
[INFO] Rolling features built (windows=(3, 5, 10))
[INFO] Opponent rolling features attached
[INFO] H2H features attached
[INFO] Context features: ['Possession', 'Shot_Creating_Actions', 'Successful_Dribbles', 'Final_Third_Entries', 'Final_Third_Entries_Allowed', 'Aerial_Battles_Won_Pct', 'Save_Pct', 'PPDA', 'Allowed_PPDA']
[INFO] Referee bias features attached
[INFO] Match-context features attached
[INFO] Dynamic motivation scores calculated
[INFO] Fatigue scores attached from 920 competition rows
[INFO] Rolling features built (windows=(3, 5, 10))
[INFO] Opponent rolling features attached
[INFO] H2H features attached
[INFO] Context features: ['Possession', 'Shot_Creating_Actions', 'Successful_Dribbles', 'Final_Third_Entries', 'Final_Third_Entries_Allowed', 'Aerial_Battles_Won_Pct', 'S

## Results Summary

In [38]:
results_df = (
    pd.DataFrame(results)
    .sort_values("test_accuracy", ascending=False)
    .reset_index(drop=True)
)
results_df

,model,validation_accuracy,test_accuracy
0,XGBoostClassifier (boosting),0.689769,0.644495
1,LogisticRegression (linear model),0.651815,0.642202
2,DecisionTreeClassifier (rule-based),0.623762,0.623853
3,RandomForestClassifier (bagging ensemble),0.645215,0.612385
4,SVM (margin-based),0.508251,0.415138
5,MLPClassifier (neural network),0.533003,0.366972
